# 从零实现 GraphSAGE：均值聚合、固定采样与归纳新节点

本 Notebook 只使用 PyTorch 基础张量与 `nn.Module`，手写 mean aggregator、`SAGEConv` 和两层 `SAGENet`；不使用 PyG、DGL、`torch_geometric` 或现成 GNN 层。覆盖完整邻居与固定 fanout 采样、mask 训练、梯度、未见节点、孤立点、tenant 泄漏反例、制品指纹和推理合同。

数据为固定种子的虚构服务关系，强制 CPU/单线程。这里验证实现与边界，不复现论文基准，也不代表生产大图性能。

In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import time  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 2601  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
DTYPE = torch.float32  # 计算并保存当前步骤的中间状态。

def canonical_fingerprint(payload) -> str:  # 定义本节可复用的核心函数。
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]  # 返回当前分支计算出的结果。

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert torch.initial_seed() == SEED  # 用受控断言验证关键不变量。
assert not any(name in globals() for name in ("torch_geometric", "dgl"))  # 用受控断言验证关键不变量。

## 1. 授权数据与局部索引

tenant-a 有 20 个基础节点，两类各 10 个；同类环和二跳关系形成主要邻域，并保留两条跨类边。tenant-b 有极端特征，原始边集中还故意包含一条跨 tenant 边。只有可信 `AuthContext` 能选择分区；局部 edge index 在过滤之后重建，不能把全局编号直接交给模型。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AuthContext:  # 定义承载本节状态与行为的数据结构。
    tenant: str  # 执行当前语句以推进本节示例。
    scopes: frozenset[str]  # 执行当前语句以推进本节示例。
    principal: str  # 执行当前语句以推进本节示例。
    def require(self, scope: str) -> None:  # 定义本节可复用的核心函数。
        if scope not in self.scopes:  # 按当前条件选择后续控制路径。
            raise PermissionError(f"缺少 scope: {scope}")  # 遇到非法合同立即显式失败。

auth_a = AuthContext("tenant-a", frozenset({"graph:read","model:predict"}), "alice")  # 计算并保存当前步骤的中间状态。
node_ids_all = [f"tenant-a:n{i:02d}" for i in range(20)] + ["tenant-b:x0","tenant-b:x1"]  # 计算并保存当前步骤的中间状态。
tenant_all = ["tenant-a"] * 20 + ["tenant-b"] * 2  # 计算并保存当前步骤的中间状态。
generator = torch.Generator().manual_seed(SEED)  # 计算并保存当前步骤的中间状态。
base0, base1 = torch.tensor([1.1,0.2,0.4,1.0]), torch.tensor([0.2,1.1,0.7,1.0])  # 计算并保存当前步骤的中间状态。
X_base = torch.stack([(base0 if i < 10 else base1) + 0.24 * torch.randn(4, generator=generator) * torch.tensor([1,1,1,0]) for i in range(20)])  # 计算并保存当前步骤的中间状态。
X_all = torch.cat([X_base, torch.tensor([[80.,80.,80.,1.],[-80.,-80.,-80.,1.]])])  # 计算并保存当前步骤的中间状态。
y_all = torch.tensor([0]*10 + [1]*10 + [-1,-1], dtype=torch.long)  # 计算并保存当前步骤的中间状态。

pair_set = set()  # 计算并保存当前步骤的中间状态。
for offset in (0,10):  # 遍历输入元素以累积或检查结果。
    for j in range(10):  # 遍历输入元素以累积或检查结果。
        for step in (1,2):  # 遍历输入元素以累积或检查结果。
            pair_set.add(tuple(sorted((offset+j, offset+(j+step)%10))))  # 执行当前语句以推进本节示例。
raw_pairs = sorted(pair_set | {(0,10),(5,15),(20,21),(0,20)})  # 计算并保存当前步骤的中间状态。

def authorize(auth: AuthContext):  # 定义本节可复用的核心函数。
    auth.require("graph:read")  # 执行当前语句以推进本节示例。
    visible = [i for i,t in enumerate(tenant_all) if t == auth.tenant]  # 计算并保存当前步骤的中间状态。
    remap = {old:new for new,old in enumerate(visible)}  # 计算并保存当前步骤的中间状态。
    pairs = [(remap[u],remap[v]) for u,v in raw_pairs if u in remap and v in remap and tenant_all[u] == tenant_all[v] == auth.tenant]  # 计算并保存当前步骤的中间状态。
    return [node_ids_all[i] for i in visible], X_all[visible].clone(), y_all[visible].clone(), sorted(pairs)  # 返回当前分支计算出的结果。

node_ids, X, y, undirected_pairs = authorize(auth_a)  # 计算并保存当前步骤的中间状态。
assert X.shape == (20,4) and y.shape == (20,)  # 用受控断言验证关键不变量。
assert len(undirected_pairs) == 42  # 用受控断言验证关键不变量。
assert all(n.startswith("tenant-a:") for n in node_ids)  # 用受控断言验证关键不变量。
assert set(y.tolist()) == {0,1}  # 用受控断言验证关键不变量。

## 2. Edge index 方向合同

本例约定 `edge_index[0]=source`、`edge_index[1]=target`，每条无向边展开为两个方向。目标节点聚合所有指向它的 source。GraphSAGE 的 self 分支独立存在，所以邻居集合不额外加入 self-loop；若混入自环，就会把自身同时计入 self 与 neighbor 两条路径。

In [ ]:
def directed_edge_index(num_nodes: int, pairs: list[tuple[int,int]]) -> torch.Tensor:  # 定义本节可复用的核心函数。
    directed = []  # 计算并保存当前步骤的中间状态。
    for u,v in pairs:  # 遍历输入元素以累积或检查结果。
        if u == v or not (0 <= u < num_nodes and 0 <= v < num_nodes):  # 按当前条件选择后续控制路径。
            raise ValueError("边端点非法或含自环")  # 遇到非法合同立即显式失败。
        directed.extend([(u,v),(v,u)])  # 执行当前语句以推进本节示例。
    directed = sorted(set(directed), key=lambda p:(p[1],p[0]))  # 计算并保存当前步骤的中间状态。
    return torch.tensor(directed, dtype=torch.long).T.contiguous()  # 返回当前分支计算出的结果。

edge_index = directed_edge_index(len(node_ids), undirected_pairs)  # 计算并保存当前步骤的中间状态。
edge_set = set(map(tuple, edge_index.T.tolist()))  # 计算并保存当前步骤的中间状态。
assert edge_index.shape == (2, 84)  # 用受控断言验证关键不变量。
assert all((v,u) in edge_set for u,v in edge_set)  # 用受控断言验证关键不变量。
assert all(u != v for u,v in edge_set)  # 用受控断言验证关键不变量。
assert int(edge_index.min()) == 0 and int(edge_index.max()) == 19  # 用受控断言验证关键不变量。

## 3. Mask 与任务边界

每类前 4 个节点训练、中间 3 个验证、最后 3 个测试。基础任务是 transductive mask 训练：全体基础节点的无标签特征和边参与传播，但损失只读取 train 标签。后面再加入训练时完全不存在的新节点，验证同一组可学习聚合参数无需重新训练即可生成表示，这才是模型结构的 inductive 能力。

In [ ]:
train_mask = torch.zeros(20,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
val_mask = torch.zeros(20,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
test_mask = torch.zeros(20,dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
for cls in (0,1):  # 遍历输入元素以累积或检查结果。
    idx = torch.where(y == cls)[0]  # 计算并保存当前步骤的中间状态。
    train_mask[idx[:4]]=True; val_mask[idx[4:7]]=True; test_mask[idx[7:]]=True  # 计算并保存当前步骤的中间状态。
cover = train_mask.to(torch.int8)+val_mask.to(torch.int8)+test_mask.to(torch.int8)  # 计算并保存当前步骤的中间状态。
assert (int(train_mask.sum()),int(val_mask.sum()),int(test_mask.sum())) == (8,6,6)  # 用受控断言验证关键不变量。
assert torch.equal(cover,torch.ones_like(cover))  # 用受控断言验证关键不变量。
assert set(y[train_mask].tolist()) == set(y[val_mask].tolist()) == set(y[test_mask].tolist()) == {0,1}  # 用受控断言验证关键不变量。
assert not torch.any(train_mask & val_mask) and not torch.any(val_mask & test_mask)  # 用受控断言验证关键不变量。

## 4. Mean aggregator 的公式与孤立点

对目标节点 (v)，邻居均值为 (m_v=|N(v)|^{-1}\sum_{u\in N(v)}h_u)。若 (N(v)=\varnothing)，本合同令 (m_v=0)，而不是 NaN 或偷偷复制 self。使用 `index_add_` 累加 source 特征和计数：

[
h'_v=W_{self}h_v+W_{neigh}m_v+b
]

输入 `x:(N,F_in)`，均值仍是 `(N,F_in)`，层输出为 `(N,F_out)`。

In [ ]:
def mean_neighbors(x: torch.Tensor, edges: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:  # 定义本节可复用的核心函数。
    if x.ndim != 2 or edges.ndim != 2 or edges.shape[0] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("x 或 edge_index shape 非法")  # 遇到非法合同立即显式失败。
    if edges.numel() and (int(edges.min()) < 0 or int(edges.max()) >= x.shape[0]):  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 越界")  # 遇到非法合同立即显式失败。
    source, target = edges[0], edges[1]  # 计算并保存当前步骤的中间状态。
    sums = torch.zeros_like(x)  # 计算并保存当前步骤的中间状态。
    counts = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)  # 计算并保存当前步骤的中间状态。
    if source.numel():  # 按当前条件选择后续控制路径。
        sums.index_add_(0, target, x[source])  # 执行当前语句以推进本节示例。
        counts.index_add_(0, target, torch.ones_like(target, dtype=x.dtype))  # 计算并保存当前步骤的中间状态。
    means = sums / counts.clamp_min(1).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
    return means, counts  # 返回当前分支计算出的结果。

full_mean, full_counts = mean_neighbors(X, edge_index)  # 计算并保存当前步骤的中间状态。
manual_n0 = X[edge_index[0, edge_index[1] == 0]].mean(dim=0)  # 计算并保存当前步骤的中间状态。
assert full_mean.shape == X.shape and full_counts.shape == (20,)  # 用受控断言验证关键不变量。
assert torch.allclose(full_mean[0], manual_n0, atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.all(full_counts > 0)  # 用受控断言验证关键不变量。
assert torch.isfinite(full_mean).all()  # 用受控断言验证关键不变量。

## 5. 手写 `SAGEConv`

self 与 neighbor 使用不同线性映射；neighbor 分支承担 bias，避免重复 bias。参数量为 (2F_{in}F_{out}+F_{out})。层本身不执行激活或归一化，方便网络明确决定层间策略，也便于单测孤立点的精确 fallback。

In [ ]:
class SAGEConv(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features: int, out_features: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_features <= 0 or out_features <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("特征维度必须为正")  # 遇到非法合同立即显式失败。
        self.in_features, self.out_features = in_features, out_features  # 计算并保存当前步骤的中间状态。
        self.self_linear = nn.Linear(in_features, out_features, bias=False)  # 计算并保存当前步骤的中间状态。
        self.neigh_linear = nn.Linear(in_features, out_features, bias=True)  # 计算并保存当前步骤的中间状态。
        nn.init.xavier_uniform_(self.self_linear.weight)  # 执行当前语句以推进本节示例。
        nn.init.xavier_uniform_(self.neigh_linear.weight)  # 执行当前语句以推进本节示例。
        nn.init.zeros_(self.neigh_linear.bias)  # 执行当前语句以推进本节示例。

    def forward(self, x: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.in_features:  # 按当前条件选择后续控制路径。
            raise ValueError("x shape 不匹配")  # 遇到非法合同立即显式失败。
        neighbor_mean, _ = mean_neighbors(x, edges)  # 计算并保存当前步骤的中间状态。
        out = self.self_linear(x) + self.neigh_linear(neighbor_mean)  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(out).all():  # 按当前条件选择后续控制路径。
            raise ValueError("SAGEConv 产生非有限值")  # 遇到非法合同立即显式失败。
        return out  # 返回当前分支计算出的结果。

probe_conv = SAGEConv(4,6)  # 计算并保存当前步骤的中间状态。
probe = probe_conv(X,edge_index)  # 计算并保存当前步骤的中间状态。
assert probe.shape == (20,6)  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in probe_conv.parameters()) == 2*4*6+6  # 用受控断言验证关键不变量。
assert probe_conv.self_linear.bias is None  # 用受控断言验证关键不变量。
assert torch.isfinite(probe).all()  # 用受控断言验证关键不变量。

## 6. 两层 `SAGENet`

第一层 4→8，ReLU 后做 L2 归一化，再 Dropout；第二层 8→2 输出 logits。L2 归一化是本教学实现的显式选择，不是 mean GraphSAGE 的强制组成。总参数量为 `(2*4*8+8)+(2*8*2+2)=106`。

In [ ]:
class SAGENet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features: int, hidden: int, classes: int, dropout: float=0.12):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.conv1 = SAGEConv(in_features,hidden)  # 计算并保存当前步骤的中间状态。
        self.conv2 = SAGEConv(hidden,classes)  # 计算并保存当前步骤的中间状态。
        self.dropout = float(dropout)  # 计算并保存当前步骤的中间状态。

    def forward(self,x:torch.Tensor,edges:torch.Tensor,return_hidden:bool=False):  # 定义本节可复用的核心函数。
        h = F.relu(self.conv1(x,edges))  # 计算并保存当前步骤的中间状态。
        h = F.normalize(h,p=2,dim=1,eps=1e-12)  # 计算并保存当前步骤的中间状态。
        h_drop = F.dropout(h,p=self.dropout,training=self.training)  # 计算并保存当前步骤的中间状态。
        logits = self.conv2(h_drop,edges)  # 计算并保存当前步骤的中间状态。
        return (logits,h) if return_hidden else logits  # 返回当前分支计算出的结果。

model = SAGENet(4,8,2)  # 计算并保存当前步骤的中间状态。
model.eval(); logits,hidden=model(X,edge_index,return_hidden=True)  # 计算并保存当前步骤的中间状态。
assert logits.shape == (20,2) and hidden.shape == (20,8)  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in model.parameters()) == 106  # 用受控断言验证关键不变量。
assert torch.allclose(hidden.norm(dim=1),torch.ones(20),atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.isfinite(logits).all()  # 用受控断言验证关键不变量。

## 7. 固定邻居采样

大图不能每层展开全部邻居。采样器按 target 建立排序邻接表，对每个 target 使用由 `seed + target` 派生的独立生成器选择至多 `fanout` 个 source，因此结果不依赖字典插入顺序；同 seed 必须逐位一致。这里只演示单层固定采样，不冒充生产的多层 computation graph/block sampler。

“不依赖输入边顺序”必须由反例锁定，而不能只从实现阅读得出。测试会固定同一个 seed，将 `edge_index` 随机重排后再次采样，并要求规范化后的结果逐位一致。


In [ ]:
def sample_neighbors(edges: torch.Tensor, num_nodes: int, fanout: int, seed: int) -> torch.Tensor:  # 定义本节可复用的核心函数。
    if fanout <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("fanout 必须为正")  # 遇到非法合同立即显式失败。
    by_target = [[] for _ in range(num_nodes)]  # 计算并保存当前步骤的中间状态。
    for source,target in edges.T.tolist():  # 遍历输入元素以累积或检查结果。
        by_target[target].append(source)  # 执行当前语句以推进本节示例。
    sampled = []  # 计算并保存当前步骤的中间状态。
    for target,sources in enumerate(by_target):  # 遍历输入元素以累积或检查结果。
        sources = sorted(set(sources))  # 计算并保存当前步骤的中间状态。
        if len(sources) > fanout:  # 按当前条件选择后续控制路径。
            generator = torch.Generator().manual_seed(seed + 1009*target)  # 计算并保存当前步骤的中间状态。
            chosen_idx = torch.randperm(len(sources),generator=generator)[:fanout].tolist()  # 计算并保存当前步骤的中间状态。
            sources = sorted(sources[i] for i in chosen_idx)  # 计算并保存当前步骤的中间状态。
        sampled.extend((source,target) for source in sources)  # 执行当前语句以推进本节示例。
    if not sampled:  # 按当前条件选择后续控制路径。
        return torch.empty((2,0),dtype=torch.long)  # 返回当前分支计算出的结果。
    return torch.tensor(sampled,dtype=torch.long).T.contiguous()  # 返回当前分支计算出的结果。

sampled_edges = sample_neighbors(edge_index,20,fanout=2,seed=SEED)  # 计算并保存当前步骤的中间状态。
sampled_again = sample_neighbors(edge_index,20,fanout=2,seed=SEED)  # 计算并保存当前步骤的中间状态。
_,sampled_counts = mean_neighbors(X,sampled_edges)  # 计算并保存当前步骤的中间状态。
assert torch.equal(sampled_edges,sampled_again)  # 用受控断言验证关键不变量。
assert int(sampled_counts.max()) <= 2 and int(sampled_counts.min()) >= 1  # 用受控断言验证关键不变量。
assert sampled_edges.shape[1] < edge_index.shape[1]  # 用受控断言验证关键不变量。
assert set(map(tuple,sampled_edges.T.tolist())) <= edge_set  # 用受控断言验证关键不变量。
edge_permutation = torch.randperm(edge_index.shape[1], generator=torch.Generator().manual_seed(SEED + 99))  # 计算并保存当前步骤的中间状态。
permuted_sample = sample_neighbors(edge_index[:, edge_permutation], 20, fanout=2, seed=SEED)  # 计算并保存当前步骤的中间状态。
assert torch.equal(permuted_sample, sampled_edges), "采样结果不应依赖 edge_index 输入顺序"  # 用受控断言验证关键不变量。


## 8. Full 与 sampled 的可验证关系

当 fanout 不小于最大度数时，采样器必须保留完整边集合，模型输出应与 full-neighbor forward 数值一致。fanout=2 时输出可以不同，这不是错误，而是带方差的近似；必须把 fanout、seed、采样器版本写进训练和制品合同。

In [ ]:
max_degree = int(full_counts.max())  # 计算并保存当前步骤的中间状态。
complete_sample = sample_neighbors(edge_index,20,fanout=max_degree,seed=SEED)  # 计算并保存当前步骤的中间状态。
assert set(map(tuple,complete_sample.T.tolist())) == edge_set  # 用受控断言验证关键不变量。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    full_logits = model(X,edge_index)  # 计算并保存当前步骤的中间状态。
    complete_logits = model(X,complete_sample)  # 计算并保存当前步骤的中间状态。
    small_sample_logits = model(X,sampled_edges)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(full_logits,complete_logits,atol=1e-7)  # 用受控断言验证关键不变量。
assert small_sample_logits.shape == full_logits.shape  # 用受控断言验证关键不变量。
assert torch.isfinite(small_sample_logits).all()  # 用受控断言验证关键不变量。
assert float((small_sample_logits-full_logits).abs().max()) > 0  # 用受控断言验证关键不变量。

## 9. 固定采样训练与 validation checkpoint

训练使用固定 `fanout=2/seed=2601` 的 edge index，使示例完全可复现；validation 也使用同一采样视图，避免每轮评估噪声改变 checkpoint。生产通常每轮重采样来扩大覆盖，但必须保存 RNG 状态、分布式 worker 规则并用多 seed 报告。test 在 checkpoint 冻结后同时报告 sampled 与 full 两种推理。

训练验收同时保存初始化参数与 eval-mode 初始 train loss。恢复 validation 选出的 checkpoint 后，要求 state_dict 至少一个张量改变、train loss 显著下降且 train-mask accuracy 达标；否则仅有有限梯度并不能证明 `optimizer.step()` 真正生效。


In [ ]:
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
model = SAGENet(4, 8, 2, dropout=0.12)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model.parameters(), lr=0.035, weight_decay=4e-4)  # 计算并保存当前步骤的中间状态。
initial_train_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_train_eval_loss = float(F.cross_entropy(model(X, sampled_edges)[train_mask], y[train_mask]))  # 计算并保存当前步骤的中间状态。

best_val, best_epoch, best_state = math.inf, -1, None  # 计算并保存当前步骤的中间状态。
history = []; started = time.perf_counter()  # 计算并保存当前步骤的中间状态。
for epoch in range(110):  # 遍历输入元素以累积或检查结果。
    model.train(); optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model(X, sampled_edges)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits[train_mask], y[train_mask])  # 计算并保存当前步骤的中间状态。
    loss.backward(); optimizer.step()  # 执行当前语句以推进本节示例。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        val_loss = float(F.cross_entropy(model(X, sampled_edges)[val_mask], y[val_mask]))  # 计算并保存当前步骤的中间状态。
    history.append((float(loss.detach()), val_loss))  # 执行当前语句以推进本节示例。
    if val_loss < best_val:  # 按当前条件选择后续控制路径。
        best_val, best_epoch = val_loss, epoch  # 计算并保存当前步骤的中间状态。
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
train_seconds = time.perf_counter() - started  # 计算并保存当前步骤的中间状态。

assert best_state is not None and 0 <= best_epoch < 110  # 用受控断言验证关键不变量。
assert len(history) == 110 and np.isfinite(np.asarray(history)).all()  # 用受控断言验证关键不变量。
assert train_seconds < 20  # 用受控断言验证关键不变量。
model.load_state_dict(best_state, strict=True); model.eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    restored_train_logits = model(X, sampled_edges)  # 计算并保存当前步骤的中间状态。
    restored_train_eval_loss = float(F.cross_entropy(restored_train_logits[train_mask], y[train_mask]))  # 计算并保存当前步骤的中间状态。
    restored_train_accuracy = float((restored_train_logits[train_mask].argmax(1) == y[train_mask]).float().mean())  # 计算并保存当前步骤的中间状态。
updated_parameter_keys = [key for key, value in model.state_dict().items()  # 计算并保存当前步骤的中间状态。
                          if not torch.equal(value.detach(), initial_train_state[key])]  # 按当前条件选择后续控制路径。
assert updated_parameter_keys, "optimizer 没有改变任何 state_dict 张量"  # 用受控断言验证关键不变量。
assert restored_train_eval_loss < initial_train_eval_loss * 0.75  # 用受控断言验证关键不变量。
assert restored_train_accuracy >= 0.90  # 用受控断言验证关键不变量。


## 10. 梯度与基础测试指标

恢复 checkpoint 后执行一次不更新参数的 backward，确认 self/neigh 两条路径及两层参数都有有限非零梯度。随后只打开一次 test mask；sampled/full 差异应作为部署选择报告，不能挑分数高的一种冒充统一结果。

In [ ]:
model.train(); optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
check_loss=F.cross_entropy(model(X,sampled_edges)[train_mask],y[train_mask]); check_loss.backward()  # 计算并保存当前步骤的中间状态。
grad_norms={name:float(p.grad.norm()) for name,p in model.named_parameters()}  # 计算并保存当前步骤的中间状态。
assert len(grad_norms)==6  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) and v>0 for v in grad_norms.values())  # 用受控断言验证关键不变量。
assert sum(grad_norms.values())>0  # 用受控断言验证关键不变量。
model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    pred_sample=model(X,sampled_edges).argmax(1)  # 计算并保存当前步骤的中间状态。
    pred_full=model(X,edge_index).argmax(1)  # 计算并保存当前步骤的中间状态。
sampled_test_accuracy=float((pred_sample[test_mask]==y[test_mask]).float().mean())  # 计算并保存当前步骤的中间状态。
full_test_accuracy=float((pred_full[test_mask]==y[test_mask]).float().mean())  # 计算并保存当前步骤的中间状态。
assert 0<=sampled_test_accuracy<=1 and 0<=full_test_accuracy<=1  # 用受控断言验证关键不变量。
assert pred_sample.shape==pred_full.shape==y.shape  # 用受控断言验证关键不变量。

## 11. 真正未见的新节点

训练结束后追加两个有邻居的新节点和一个完全孤立节点。它们从未出现在 optimizer、mask 或训练 edge index 中；模型根据同一 feature schema 和当前邻居即时生成表示。GraphSAGE 的“inductive”指聚合函数可复用，并不保证新域、新特征分布或新关系类型上的效果。

仅检查新节点输出 finite 仍然可能掩盖“新节点边被丢弃”。因此再构造两个**自身特征完全相同、但各自只有一个不同邻居**的新节点，固定 `SAGEConv` 权重，让预期 neighbor mean 与输出都能手算；输出差异只能来自邻域。


In [ ]:
new_ids=["tenant-a:new-front","tenant-a:new-back","tenant-a:new-isolated"]  # 计算并保存当前步骤的中间状态。
new_X=torch.tensor([[1.0,0.25,0.45,1.0],[0.25,1.0,0.65,1.0],[0.6,0.6,0.5,1.0]],dtype=DTYPE)  # 计算并保存当前步骤的中间状态。
X_extended=torch.cat([X,new_X],dim=0)  # 计算并保存当前步骤的中间状态。
extended_pairs=undirected_pairs+[(20,0),(20,1),(20,2),(21,10),(21,11),(21,12)]  # 计算并保存当前步骤的中间状态。
extended_edges=directed_edge_index(23,extended_pairs)  # 计算并保存当前步骤的中间状态。
state_before={k:v.detach().clone() for k,v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    extended_logits,extended_hidden=model(X_extended,extended_edges,return_hidden=True)  # 计算并保存当前步骤的中间状态。
state_after=model.state_dict()  # 计算并保存当前步骤的中间状态。
assert extended_logits.shape==(23,2) and extended_hidden.shape==(23,8)  # 用受控断言验证关键不变量。
assert all(torch.equal(state_before[k],state_after[k]) for k in state_before)  # 用受控断言验证关键不变量。
assert all(new_id not in node_ids for new_id in new_ids)  # 用受控断言验证关键不变量。
assert torch.isfinite(extended_logits[20:]).all()  # 用受控断言验证关键不变量。
inductive_mean, inductive_counts = mean_neighbors(X_extended, extended_edges)  # 计算并保存当前步骤的中间状态。
assert int(inductive_counts[20]) == 3 and int(inductive_counts[21]) == 3  # 用受控断言验证关键不变量。
assert not torch.allclose(inductive_mean[20], inductive_mean[21])  # 用受控断言验证关键不变量。

# 精确 inductive oracle：节点 2/3 的 self feature 相同，邻居分别为节点 0/1。
oracle_X26 = torch.tensor([  # 计算并保存当前步骤的中间状态。
    [1., 0., 0., 0.], [0., 1., 0., 0.],  # 执行当前语句以推进本节示例。
    [0.5, 0.5, 0.5, 1.], [0.5, 0.5, 0.5, 1.],  # 执行当前语句以推进本节示例。
])  # 执行当前语句以推进本节示例。
oracle_edges26 = torch.tensor([[0, 1], [2, 3]], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
oracle_mean26, oracle_counts26 = mean_neighbors(oracle_X26, oracle_edges26)  # 计算并保存当前步骤的中间状态。
oracle_conv26 = SAGEConv(4, 2)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    oracle_conv26.self_linear.weight.zero_()  # 执行当前语句以推进本节示例。
    oracle_conv26.neigh_linear.weight.zero_()  # 执行当前语句以推进本节示例。
    oracle_conv26.neigh_linear.weight[0, 0] = 1  # 计算并保存当前步骤的中间状态。
    oracle_conv26.neigh_linear.weight[1, 1] = 1  # 计算并保存当前步骤的中间状态。
    oracle_conv26.neigh_linear.bias.zero_()  # 执行当前语句以推进本节示例。
    oracle_output26 = oracle_conv26(oracle_X26, oracle_edges26)  # 计算并保存当前步骤的中间状态。
assert torch.equal(oracle_X26[2], oracle_X26[3])  # 用受控断言验证关键不变量。
assert oracle_counts26.tolist() == [0.0, 0.0, 1.0, 1.0]  # 用受控断言验证关键不变量。
assert torch.equal(oracle_mean26[2], oracle_X26[0])  # 用受控断言验证关键不变量。
assert torch.equal(oracle_mean26[3], oracle_X26[1])  # 用受控断言验证关键不变量。
assert torch.equal(oracle_output26[2], torch.tensor([1., 0.]))  # 用受控断言验证关键不变量。
assert torch.equal(oracle_output26[3], torch.tensor([0., 1.]))  # 用受控断言验证关键不变量。
assert not torch.equal(oracle_output26[2], oracle_output26[3])  # 用受控断言验证关键不变量。


## 12. 孤立点 fallback

新节点 22 没有入边；mean aggregator 必须返回零向量，第一层结果严格等于 self 线性分支加 neighbor bias。它仍可依赖自身属性分类，但响应要标记 `isolated/cold_start`，不能把缺少邻居解释成低风险或高置信证据。

In [ ]:
extended_mean,extended_counts=mean_neighbors(X_extended,extended_edges)  # 计算并保存当前步骤的中间状态。
isolated=22  # 计算并保存当前步骤的中间状态。
first_layer=model.conv1(X_extended,extended_edges)  # 计算并保存当前步骤的中间状态。
expected_isolated=model.conv1.self_linear(X_extended[isolated])+model.conv1.neigh_linear.bias  # 计算并保存当前步骤的中间状态。
assert int(extended_counts[isolated])==0  # 用受控断言验证关键不变量。
assert torch.allclose(extended_mean[isolated],torch.zeros(4))  # 用受控断言验证关键不变量。
assert torch.allclose(first_layer[isolated],expected_isolated,atol=1e-7)  # 用受控断言验证关键不变量。
assert int(extended_logits[isolated].argmax()) in {0,1}  # 用受控断言验证关键不变量。

## 13. 制品绑定：训练图与归纳请求图要区分

GraphSAGE 制品绑定**训练图指纹**、feature schema、采样配置和 state_dict；它不能要求每个归纳请求图都等于训练图，否则新节点永远无法进入。在线响应另算当前授权请求图指纹并记录在 trace。内容哈希防错配，生产还需要签名、注册表权限和不可变存储。

训练制品还必须绑定训练期**有序特征快照**：`snapshot_id/as_of/node_order/shape/dtype/content_sha256`。训练图拓扑相同但任意特征值变化时，重新计算的快照指纹必须使验证失败。归纳请求拥有独立的可信快照与独立 trace，不会被错误要求等于训练快照。


In [ ]:
FEATURE_SCHEMA = {"order": ["http_ratio","batch_ratio","cpu_norm","bias"],  # 计算并保存当前步骤的中间状态。
                  "dtype": "float32", "source": "synthetic-v1", "fit_split": "not_applicable"}  # 执行当前语句以推进本节示例。
FEATURE_SCHEMA_ID = canonical_fingerprint(FEATURE_SCHEMA)  # 计算并保存当前步骤的中间状态。
TRAIN_FEATURE_SNAPSHOT_ID = "tenant-a-train-features-2026-07-01T00:00:00Z"  # 计算并保存当前步骤的中间状态。
TRAIN_FEATURE_AS_OF = "2026-07-01T00:00:00Z"  # 计算并保存当前步骤的中间状态。

def feature_snapshot_descriptor(ordered_node_ids, features, snapshot_id: str, as_of: str) -> dict:  # 定义本节可复用的核心函数。
    if features.ndim != 2 or len(ordered_node_ids) != features.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("节点顺序与特征 shape 不匹配")  # 遇到非法合同立即显式失败。
    if len(set(ordered_node_ids)) != len(ordered_node_ids) or not torch.isfinite(features).all():  # 按当前条件选择后续控制路径。
        raise ValueError("节点 ID 必须唯一且特征必须有限")  # 遇到非法合同立即显式失败。
    value = features.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
    return {"snapshot_id": snapshot_id, "as_of": as_of,  # 返回当前分支计算出的结果。
            "node_order": list(ordered_node_ids), "shape": list(value.shape),  # 执行当前语句以推进本节示例。
            "dtype": str(value.dtype),  # 执行当前语句以推进本节示例。
            "content_sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest()}  # 执行当前语句以推进本节示例。

TRAIN_FEATURE_SNAPSHOT = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
    node_ids, X, TRAIN_FEATURE_SNAPSHOT_ID, TRAIN_FEATURE_AS_OF  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
TRAIN_FEATURE_SNAPSHOT_FINGERPRINT = canonical_fingerprint(TRAIN_FEATURE_SNAPSHOT)  # 计算并保存当前步骤的中间状态。
train_graph_payload = {  # 计算并保存当前步骤的中间状态。
    "tenant": auth_a.tenant, "as_of": "2026-07-01T00:00:00Z", "nodes": node_ids,  # 执行当前语句以推进本节示例。
    "edges": sorted([sorted((node_ids[u],node_ids[v])) for u,v in undirected_pairs]),  # 执行当前语句以推进本节示例。
    "edge_direction": "undirected-expanded-source-to-target-v1",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
TRAIN_GRAPH_FINGERPRINT = canonical_fingerprint(train_graph_payload)  # 计算并保存当前步骤的中间状态。

def state_dict_fingerprint(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()[:20]  # 返回当前分支计算出的结果。

artifact = {  # 计算并保存当前步骤的中间状态。
    "model_type": "SAGENet-from-scratch", "model_version": "sage-v2",  # 执行当前语句以推进本节示例。
    "tenant": auth_a.tenant,  # 执行当前语句以推进本节示例。
    "training_graph_fingerprint": TRAIN_GRAPH_FINGERPRINT,  # 执行当前语句以推进本节示例。
    "feature_schema_id": FEATURE_SCHEMA_ID,  # 执行当前语句以推进本节示例。
    "training_feature_snapshot_id": TRAIN_FEATURE_SNAPSHOT_ID,  # 执行当前语句以推进本节示例。
    "training_feature_snapshot_fingerprint": TRAIN_FEATURE_SNAPSHOT_FINGERPRINT,  # 执行当前语句以推进本节示例。
    "architecture": {"dims": [4,8,2], "aggregator": "mean", "hidden_l2_normalize": True,  # 执行当前语句以推进本节示例。
                     "dropout": 0.12},  # 执行当前语句以推进本节示例。
    "training_sampler": {"name": "fixed-uniform-v1", "fanout": 2, "seed": SEED},  # 执行当前语句以推进本节示例。
    "inference_aggregation": "trusted-request-graph-full-neighbor-v1",  # 执行当前语句以推进本节示例。
    "state_dict_fingerprint": state_dict_fingerprint(model.state_dict()),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact["artifact_id"] = canonical_fingerprint(artifact)  # 计算并保存当前步骤的中间状态。

def validate_artifact(value, state, current_training_feature_snapshot):  # 定义本节可复用的核心函数。
    unsigned = {k:v for k,v in value.items() if k != "artifact_id"}  # 计算并保存当前步骤的中间状态。
    if canonical_fingerprint(unsigned) != value.get("artifact_id"):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact hash 不匹配")  # 遇到非法合同立即显式失败。
    if (value.get("training_graph_fingerprint") != TRAIN_GRAPH_FINGERPRINT or  # 按当前条件选择后续控制路径。
            value.get("feature_schema_id") != FEATURE_SCHEMA_ID):  # 计算并保存当前步骤的中间状态。
        raise ValueError("训练图或特征 schema 不匹配")  # 遇到非法合同立即显式失败。
    current_feature_fp = canonical_fingerprint(current_training_feature_snapshot)  # 计算并保存当前步骤的中间状态。
    if (value.get("training_feature_snapshot_id") != current_training_feature_snapshot.get("snapshot_id") or  # 按当前条件选择后续控制路径。
            value.get("training_feature_snapshot_fingerprint") != current_feature_fp):  # 计算并保存当前步骤的中间状态。
        raise ValueError("训练特征快照不匹配")  # 遇到非法合同立即显式失败。
    if value.get("state_dict_fingerprint") != state_dict_fingerprint(state):  # 按当前条件选择后续控制路径。
        raise ValueError("state_dict 不匹配")  # 遇到非法合同立即显式失败。
    if value.get("training_sampler") != {"name":"fixed-uniform-v1","fanout":2,"seed":SEED}:  # 按当前条件选择后续控制路径。
        raise ValueError("训练采样合同不匹配")  # 遇到非法合同立即显式失败。
    return True  # 返回当前分支计算出的结果。

assert validate_artifact(artifact, model.state_dict(), TRAIN_FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
assert len(artifact["artifact_id"]) == 20  # 用受控断言验证关键不变量。
for field, bad in (("training_graph_fingerprint","bad"),  # 遍历输入元素以累积或检查结果。
                   ("feature_schema_id","bad"),  # 执行当前语句以推进本节示例。
                   ("training_feature_snapshot_fingerprint","bad"),  # 执行当前语句以推进本节示例。
                   ("state_dict_fingerprint","bad")):  # 执行当前语句以推进本节示例。
    forged = dict(artifact); forged[field] = bad  # 计算并保存当前步骤的中间状态。
    forged["artifact_id"] = canonical_fingerprint({k:v for k,v in forged.items() if k != "artifact_id"})  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        validate_artifact(forged, model.state_dict(), TRAIN_FEATURE_SNAPSHOT)  # 执行当前语句以推进本节示例。
        raise AssertionError(f"伪造 {field} 未拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

changed_train_X = X.clone(); changed_train_X[0, 0] += 25.0  # 计算并保存当前步骤的中间状态。
changed_train_snapshot = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
    node_ids, changed_train_X, TRAIN_FEATURE_SNAPSHOT_ID, TRAIN_FEATURE_AS_OF  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert canonical_fingerprint(changed_train_snapshot) != TRAIN_FEATURE_SNAPSHOT_FINGERPRINT  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    validate_artifact(artifact, model.state_dict(), changed_train_snapshot)  # 执行当前语句以推进本节示例。
    raise AssertionError("训练特征内容改变后仍通过 artifact 校验")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "特征快照" in str(exc)  # 用受控断言验证关键不变量。


## 14. tenant 泄漏反例与归纳推理服务

原始跨 tenant 边会显著改变 node 0 的邻居均值；即使稍后裁掉 tenant-b 输出，表示已经污染。服务只接受内部授权后的节点、特征和 edge index，客户端只提交要查询的稳定 ID。返回同时携带训练图和当前请求图指纹，明确二者不必相同。

本节把“内部授权图”落实为类型和注册表，而非函数注释。公开推理入口只接收 `snapshot_id + requested node IDs`；`features/edges/node tenant metadata` 只能由持有私有 issuer token 的内部装载器注册。注册器根据可信 tenant 元数据逐节点校验，绝不把 ID 前缀当授权事实。服务还会在每次调用时重算请求图与特征快照指纹，防止冻结 dataclass 内部张量被原地污染。


In [ ]:
raw_edge_index = directed_edge_index(22, raw_pairs)  # 计算并保存当前步骤的中间状态。
unsafe_mean, _ = mean_neighbors(X_all, raw_edge_index)  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(unsafe_mean[0], full_mean[0])  # 用受控断言验证关键不变量。
assert 20 in raw_edge_index[0, raw_edge_index[1] == 0].tolist()  # 用受控断言验证关键不变量。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class TrustedGraphSnapshot:  # 定义承载本节状态与行为的数据结构。
    snapshot_id: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    as_of: str  # 执行当前语句以推进本节示例。
    node_ids: tuple[str, ...]  # 执行当前语句以推进本节示例。
    node_tenants: tuple[str, ...]  # 执行当前语句以推进本节示例。
    features: torch.Tensor  # 执行当前语句以推进本节示例。
    edges: torch.Tensor  # 执行当前语句以推进本节示例。
    graph_fingerprint: str  # 执行当前语句以推进本节示例。
    feature_snapshot_fingerprint: str  # 执行当前语句以推进本节示例。

_REGISTRY_ISSUER = object()  # 计算并保存当前步骤的中间状态。
_TRUSTED_GRAPH_REGISTRY: dict[str, TrustedGraphSnapshot] = {}  # 计算并保存当前步骤的中间状态。

def request_graph_fingerprint(snapshot_id, tenant, as_of, ids, edges):  # 定义本节可复用的核心函数。
    if edges.ndim != 2 or edges.shape[0] != 2 or edges.dtype != torch.long:  # 按当前条件选择后续控制路径。
        raise ValueError("可信快照 edge_index 必须是 long (2,E)")  # 遇到非法合同立即显式失败。
    if edges.numel() and (int(edges.min()) < 0 or int(edges.max()) >= len(ids)):  # 按当前条件选择后续控制路径。
        raise ValueError("可信快照 edge_index 越界")  # 遇到非法合同立即显式失败。
    payload = {"snapshot_id": snapshot_id, "tenant": tenant, "as_of": as_of,  # 计算并保存当前步骤的中间状态。
               "nodes": list(ids), "directed_edges": sorted(map(list, edges.T.cpu().tolist())),  # 执行当前语句以推进本节示例。
               "direction": "source-to-target"}  # 执行当前语句以推进本节示例。
    return canonical_fingerprint(payload)  # 返回当前分支计算出的结果。

def _register_trusted_snapshot(snapshot_id, tenant, as_of, ids, node_tenants,  # 定义本节可复用的核心函数。
                               features, edges, *, issuer_token):  # 执行当前语句以推进本节示例。
    if issuer_token is not _REGISTRY_ISSUER:  # 按当前条件选择后续控制路径。
        raise PermissionError("只有内部图注册器可以签发 TrustedGraphSnapshot")  # 遇到非法合同立即显式失败。
    if snapshot_id in _TRUSTED_GRAPH_REGISTRY:  # 按当前条件选择后续控制路径。
        raise ValueError("snapshot_id 已注册")  # 遇到非法合同立即显式失败。
    if len(ids) != len(node_tenants) or len(ids) != features.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("节点、tenant 元数据与特征行数不一致")  # 遇到非法合同立即显式失败。
    if len(set(ids)) != len(ids):  # 按当前条件选择后续控制路径。
        raise ValueError("可信快照节点 ID 不可重复")  # 遇到非法合同立即显式失败。
    # tenant 权威来自内部元数据，而不是可伪装的字符串前缀。
    if any(node_tenant != tenant for node_tenant in node_tenants):  # 按当前条件选择后续控制路径。
        raise PermissionError("快照包含其他 tenant 的节点元数据")  # 遇到非法合同立即显式失败。
    if any(not node.startswith(tenant + ":") for node in ids):  # 按当前条件选择后续控制路径。
        raise ValueError("稳定 ID 前缀与可信 tenant 元数据矛盾")  # 遇到非法合同立即显式失败。
    feature_copy = features.detach().cpu().contiguous().clone()  # 计算并保存当前步骤的中间状态。
    edge_copy = edges.detach().cpu().contiguous().clone()  # 计算并保存当前步骤的中间状态。
    feature_descriptor = feature_snapshot_descriptor(ids, feature_copy, snapshot_id, as_of)  # 计算并保存当前步骤的中间状态。
    graph_fp = request_graph_fingerprint(snapshot_id, tenant, as_of, ids, edge_copy)  # 计算并保存当前步骤的中间状态。
    snapshot = TrustedGraphSnapshot(  # 计算并保存当前步骤的中间状态。
        snapshot_id, tenant, as_of, tuple(ids), tuple(node_tenants),  # 执行当前语句以推进本节示例。
        feature_copy, edge_copy, graph_fp, canonical_fingerprint(feature_descriptor)  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    _TRUSTED_GRAPH_REGISTRY[snapshot_id] = snapshot  # 计算并保存当前步骤的中间状态。
    return snapshot  # 返回当前分支计算出的结果。

REQUEST_SNAPSHOT_ID = "tenant-a-request-graph-2026-07-02T00:00:00Z"  # 计算并保存当前步骤的中间状态。
trusted_extended = _register_trusted_snapshot(  # 计算并保存当前步骤的中间状态。
    REQUEST_SNAPSHOT_ID, "tenant-a", "2026-07-02T00:00:00Z",  # 执行当前语句以推进本节示例。
    node_ids + new_ids, ["tenant-a"] * len(node_ids + new_ids),  # 执行当前语句以推进本节示例。
    X_extended, extended_edges, issuer_token=_REGISTRY_ISSUER,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。

def inductive_predict(auth: AuthContext, requested: list[str], snapshot_id: str, model, artifact):  # 定义本节可复用的核心函数。
    auth.require("model:predict")  # 执行当前语句以推进本节示例。
    if not isinstance(snapshot_id, str) or snapshot_id not in _TRUSTED_GRAPH_REGISTRY:  # 按当前条件选择后续控制路径。
        raise PermissionError("未知或未注册的可信图快照")  # 遇到非法合同立即显式失败。
    snapshot = _TRUSTED_GRAPH_REGISTRY[snapshot_id]  # 计算并保存当前步骤的中间状态。
    if snapshot.tenant != auth.tenant or any(t != auth.tenant for t in snapshot.node_tenants):  # 按当前条件选择后续控制路径。
        raise PermissionError("可信快照与调用 tenant 不匹配")  # 遇到非法合同立即显式失败。
    current_training_snapshot = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
        node_ids, X, TRAIN_FEATURE_SNAPSHOT_ID, TRAIN_FEATURE_AS_OF  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    validate_artifact(artifact, model.state_dict(), current_training_snapshot)  # 执行当前语句以推进本节示例。
    current_request_feature = feature_snapshot_descriptor(  # 计算并保存当前步骤的中间状态。
        snapshot.node_ids, snapshot.features, snapshot.snapshot_id, snapshot.as_of  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    current_request_feature_fp = canonical_fingerprint(current_request_feature)  # 计算并保存当前步骤的中间状态。
    current_request_graph_fp = request_graph_fingerprint(  # 计算并保存当前步骤的中间状态。
        snapshot.snapshot_id, snapshot.tenant, snapshot.as_of, snapshot.node_ids, snapshot.edges  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    if current_request_feature_fp != snapshot.feature_snapshot_fingerprint:  # 按当前条件选择后续控制路径。
        raise RuntimeError("可信快照特征在注册后被污染")  # 遇到非法合同立即显式失败。
    if current_request_graph_fp != snapshot.graph_fingerprint:  # 按当前条件选择后续控制路径。
        raise RuntimeError("可信快照图在注册后被污染")  # 遇到非法合同立即显式失败。
    index = {node:i for i,node in enumerate(snapshot.node_ids)}  # 计算并保存当前步骤的中间状态。
    if not requested or any(node not in index for node in requested):  # 按当前条件选择后续控制路径。
        raise PermissionError("请求含未知或越权节点")  # 遇到非法合同立即显式失败。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        probs = model(snapshot.features, snapshot.edges).softmax(1)  # 计算并保存当前步骤的中间状态。
    rows = [{"node_id": node, "probabilities": probs[index[node]].tolist(),  # 计算并保存当前步骤的中间状态。
             "isolated": int((snapshot.edges[1] == index[node]).sum()) == 0}  # 计算并保存当前步骤的中间状态。
            for node in requested]  # 遍历输入元素以累积或检查结果。
    return {"predictions": rows, "trace": {  # 返回当前分支计算出的结果。
        "artifact_id": artifact["artifact_id"],  # 执行当前语句以推进本节示例。
        "training_graph_fingerprint": TRAIN_GRAPH_FINGERPRINT,  # 执行当前语句以推进本节示例。
        "training_feature_snapshot_fingerprint": TRAIN_FEATURE_SNAPSHOT_FINGERPRINT,  # 执行当前语句以推进本节示例。
        "request_snapshot_id": snapshot.snapshot_id,  # 执行当前语句以推进本节示例。
        "request_snapshot_as_of": snapshot.as_of,  # 执行当前语句以推进本节示例。
        "request_graph_fingerprint": current_request_graph_fp,  # 执行当前语句以推进本节示例。
        "request_feature_snapshot_fingerprint": current_request_feature_fp,  # 执行当前语句以推进本节示例。
        "feature_schema_id": FEATURE_SCHEMA_ID,  # 执行当前语句以推进本节示例。
    }}  # 执行当前语句以推进本节示例。

served = inductive_predict(auth_a, new_ids, REQUEST_SNAPSHOT_ID, model, artifact)  # 计算并保存当前步骤的中间状态。
assert len(served["predictions"]) == 3  # 用受控断言验证关键不变量。
assert served["predictions"][-1]["isolated"] is True  # 用受控断言验证关键不变量。
assert served["trace"]["request_graph_fingerprint"] != served["trace"]["training_graph_fingerprint"]  # 用受控断言验证关键不变量。
assert served["trace"]["request_feature_snapshot_fingerprint"] == trusted_extended.feature_snapshot_fingerprint  # 用受控断言验证关键不变量。

# fail-closed 1：未知 snapshot_id 不能夹带调用方 raw graph。
try:  # 尝试执行可能失败的受控操作。
    inductive_predict(auth_a, [node_ids[0]], "unregistered-polluted-snapshot", model, artifact)  # 执行当前语句以推进本节示例。
    raise AssertionError("未注册污染图未被拒绝")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

# fail-closed 2：ID 被伪装成 tenant-a，但可信元数据仍是 tenant-b。
camouflaged_id = "tenant-a:camouflaged-external"  # 计算并保存当前步骤的中间状态。
polluted_features = torch.cat([X, X_all[20:21]], dim=0)  # 计算并保存当前步骤的中间状态。
polluted_edges = directed_edge_index(21, undirected_pairs + [(20, 0)])  # 计算并保存当前步骤的中间状态。
assert camouflaged_id.startswith("tenant-a:")  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    _register_trusted_snapshot(  # 执行当前语句以推进本节示例。
        "polluted-camouflage", "tenant-a", "2026-07-02T00:00:00Z",  # 执行当前语句以推进本节示例。
        node_ids + [camouflaged_id], ["tenant-a"] * 20 + ["tenant-b"],  # 执行当前语句以推进本节示例。
        polluted_features, polluted_edges, issuer_token=_REGISTRY_ISSUER,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("伪装 ID 的跨 tenant 污染图未被拒绝")  # 遇到非法合同立即显式失败。
except PermissionError as exc:  # 捕获预期异常并验证失败分支。
    assert "tenant" in str(exc)  # 用受控断言验证关键不变量。
assert "polluted-camouflage" not in _TRUSTED_GRAPH_REGISTRY  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    inductive_predict(auth_a, ["tenant-b:x0"], REQUEST_SNAPSHOT_ID, model, artifact)  # 执行当前语句以推进本节示例。
    raise AssertionError("跨 tenant 请求未拒绝")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。


## 15. 复杂度与生产边界

完整 mean 聚合每层约 (O(EF))，固定 fanout 的多层展开约为 (O(B\prod_l fanout_l))，仍会随层数指数扩张。本例 Python 级采样和整图 `index_add_` 只适合小图；生产应使用版本化 CSR/CSC、批量 block sampler、分布式特征存储、worker RNG、缓存一致性和反压。

上线需分别评估 seen/unseen/isolated 节点、度数与时间切片、采样覆盖与方差、0-hop 基线、多 seed 置信区间、标签延迟、tenant 越权、模型回滚和邻居服务降级。新节点能 forward 不等于具有可靠泛化。

In [ ]:
assert isinstance(model.conv1, SAGEConv) and isinstance(model.conv2, SAGEConv)  # 用受控断言验证关键不变量。
assert model.conv1.self_linear.weight.grad is not None and model.conv2.neigh_linear.weight.grad is not None  # 用受控断言验证关键不变量。
assert state_dict_fingerprint(model.state_dict()) == artifact["state_dict_fingerprint"]  # 用受控断言验证关键不变量。
assert FEATURE_SCHEMA["order"] == ["http_ratio","batch_ratio","cpu_norm","bias"]  # 用受控断言验证关键不变量。
assert set(map(tuple, sampled_edges.T.tolist())) <= edge_set  # 用受控断言验证关键不变量。
assert torch.equal(permuted_sample, sampled_edges)  # 用受控断言验证关键不变量。
assert int(extended_counts[22]) == 0 and served["predictions"][-1]["isolated"]  # 用受控断言验证关键不变量。
assert updated_parameter_keys and restored_train_eval_loss < initial_train_eval_loss  # 用受控断言验证关键不变量。
assert artifact["training_feature_snapshot_fingerprint"] == canonical_fingerprint(TRAIN_FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
assert REQUEST_SNAPSHOT_ID in _TRUSTED_GRAPH_REGISTRY and "polluted-camouflage" not in _TRUSTED_GRAPH_REGISTRY  # 用受控断言验证关键不变量。
assert train_seconds < 20 and validate_artifact(artifact, model.state_dict(), TRAIN_FEATURE_SNAPSHOT)  # 用受控断言验证关键不变量。
print({"status":"PASS", "model":"GraphSAGE-from-scratch", "best_epoch":best_epoch,  # 执行当前语句以推进本节示例。
       "sampled_test_accuracy":round(sampled_test_accuracy,3), "full_test_accuracy":round(full_test_accuracy,3),  # 执行当前语句以推进本节示例。
       "train_loss_before_after":[round(initial_train_eval_loss,4),round(restored_train_eval_loss,4)],  # 执行当前语句以推进本节示例。
       "params":sum(p.numel() for p in model.parameters()), "seconds":round(train_seconds,3)})  # 执行当前语句以推进本节示例。


## 16. 原始与官方资料

- Hamilton, Ying & Leskovec, *Inductive Representation Learning on Large Graphs*：https://proceedings.neurips.cc/paper/2017/hash/5dd9db5e033da9c6fb5ba83c7a7ebea9-Abstract.html
- PyTorch 官方 `nn.Module` 文档：https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 官方 Module/State 文档：https://docs.pytorch.org/docs/stable/notes/modules.html
- PyTorch 官方随机种子 API：https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html

原论文用于 mean aggregator、采样与 inductive 表示背景；固定采样、tenant 前置过滤、训练图/请求图区分和制品校验是本 Notebook 的工程扩展。